In [15]:
import numpy as np
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel, VarianceThreshold, SelectPercentile
from sklearn.model_selection import RandomizedSearchCV, RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from dotenv import load_dotenv
import joblib
from gene_thesaurus import GeneThesaurus


In [16]:
load_dotenv()

X_train_path = os.getenv("X_TRAIN_CSV_PATH")
y_train_path = os.getenv("Y_TRAIN_TSV_PATH")
X_test_path = os.getenv("X_TEST_CSV_PATH")
y_test_path = os.getenv("Y_TEST_TSV_PATH")
model_path = os.getenv("RF_MODEL_PATH")
selected_features_path = os.getenv("SELECTED_FEATURES_PATH")
hyperparameters_path = os.getenv("HYPERPARAMETERS_PATH")
pipeline_path = os.getenv("PIPELINE_PATH")
random_seed = int(os.getenv("RANDOM_SEED"))
n_jobs = int(os.getenv("N_JOBS", -1))

# Print configuration
print(f"X_train_path: {X_train_path}")
print(f"y_train_path: {y_train_path}")
print(f"X_test_path: {X_test_path}")
print(f"y_test_path: {y_test_path}")
print(f"model_path: {model_path}")
print(f"selected_features_path: {selected_features_path}")
print(f"hyperparameters_path: {hyperparameters_path}")
print(f"pipeline_path: {pipeline_path}")
print(f"random_seed: {random_seed}")
print(f"n_jobs: {n_jobs}")

# Hardcoded variables
VARIANCE_THRESHOLD=0.01  # REMOVE NEAR-CONSTANT GENES
FEATURE_SELECTION_PERCENTILE=20  # KEEP TOP n% OF FEATURES
FINAL_N_ESTIMATORS=500

X_train_path: /home/mariya/Data/pallas/nf-core-gexml-output-train/transpose/transposed.csv
y_train_path: /home/mariya/Data/pallas/nf-core-gexml-output-train/class/subtypes.tsv
X_test_path: /home/mariya/Data/pallas/nf-core-gexml-output-cal/transpose/transposed.csv
y_test_path: /home/mariya/Data/pallas/nf-core-gexml-output-cal/class/subtypes.tsv
model_path: /home/mariya/Data/pallas/training/gex/gex.model.pkl
selected_features_path: /home/mariya/Data/pallas/training/gex/gex.selected_features.pkl
hyperparameters_path: /home/mariya/Data/pallas/training/gex/gex.hyperparameters.pkl
pipeline_path: /home/mariya/Data/pallas/training/gex/gex.pipeline.pkl
random_seed: 42
n_jobs: 10


In [17]:
# Read in X_train and y_train
X_train = pd.read_csv(X_train_path, index_col=0)
X_train


,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293642,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525
id,,,,,,,,,,,,,,,,,,,,,
ALLT-315,-0.808262,-1.306820,4.689345,3.839388,4.139414,5.260186,0.742914,4.232390,4.060798,5.839346,...,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682
ALLT-318,1.136039,-1.306820,4.816215,4.191404,4.232299,9.302817,4.401644,6.778864,6.549788,5.996646,...,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682
ALLT-320,0.437586,-1.306820,4.554285,3.529770,5.009472,2.291568,-0.690692,1.249735,3.258545,6.024946,...,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682
ALLT-322,-0.554216,-1.306820,3.952087,4.363253,4.395150,3.328332,1.455490,3.089793,2.920373,6.830426,...,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682
ALLT-324,0.403401,-0.855787,4.318310,3.742426,3.945380,4.256678,1.818614,4.296264,3.993554,6.472188,...,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SJPHALL020047,-0.818357,-1.306820,2.758634,4.230391,3.768532,6.199457,5.823855,4.596740,5.245347,6.772595,...,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682
SJPHALL021,-1.306820,-1.306820,4.019197,4.393308,4.131674,7.854486,5.637584,6.854045,5.423396,5.989110,...,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682
SJPHALL022668,-1.306820,-1.306820,5.335797,3.914572,3.162256,5.597064,3.117018,4.487487,4.792926,4.958000,...,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682


In [18]:
y_train = pd.read_csv(y_train_path, sep="\t", index_col=0)
y_train

,class
sample,
TARGET-20-PASUGU,"AML, CBFB::MYH11"
TCGA-AB-2825,"AML, NPM1alt"
TARGET-20-PASEGC,"AML, NPM1alt"
TARGET-20-PASINA,"AML, RUNX1::RUNX1T1"
TARGET-20-PAKLPD,"AML, KMT2A-r"
...,...
AML093-D,"AML, KMT2A-r"
AML098-D,"AML, CEBPAalt"
BM6647_II,Non-leukemic


In [19]:
# align the indices
X_train = X_train.loc[y_train.index]
X_train

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293642,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525
sample,,,,,,,,,,,,,,,,,,,,,
TARGET-20-PASUGU,-1.306820,-1.30682,4.918624,3.134621,2.808360,7.498755,2.808807,5.686262,4.417575,5.943645,...,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682
TCGA-AB-2825,-1.306820,-1.30682,3.912727,4.050077,4.178485,9.420092,3.664777,5.033878,5.821224,5.315042,...,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682
TARGET-20-PASEGC,-1.306820,-1.30682,4.383235,3.755870,3.788430,0.649241,1.653411,4.226159,4.184311,6.871042,...,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682
TARGET-20-PASINA,-1.306820,-1.30682,4.905679,2.368093,3.147617,1.498380,4.277431,4.405865,3.725803,6.971122,...,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682
TARGET-20-PAKLPD,-1.306820,-1.30682,5.559372,2.811965,1.854785,8.260069,0.738431,5.760963,4.269713,5.457689,...,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
AML093-D,-0.876077,-1.30682,4.935494,4.161708,3.166212,2.918847,0.908432,4.712349,4.508049,6.566320,...,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682
AML098-D,0.959997,-1.30682,3.782717,3.433012,4.770819,5.797580,1.002077,3.603469,6.187951,5.953336,...,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682
BM6647_II,1.277069,-1.30682,4.115642,4.313831,4.520966,3.227480,1.957367,3.626837,4.612693,6.557704,...,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682,-1.30682


In [20]:
X_test = pd.read_csv(X_test_path, index_col=0)
y_test = pd.read_csv(y_test_path, sep="\t", index_col=0)
y_test

,class
sample,
SJBALL041777,"B-ALL, BCR::ABL1-like"
SJBALL013388,"B-ALL, MEF2D-r"
SJALL049970,"B-ALL, DUX4-r"
SJBALL020583,"B-ALL, ETV6::RUNX1"
SJCOGALL010907,"B-ALL, iAMP21"
...,...
AML044-D,"AML, NPM1alt"
867_B,Non-leukemic
867_T,Non-leukemic


In [21]:
# align the indices
X_test = X_test.loc[y_test.index]
X_test

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293642,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525
sample,,,,,,,,,,,,,,,,,,,,,
SJBALL041777,-1.440565,-1.440565,5.077498,3.810629,5.200547,0.824143,-0.933615,1.034715,3.558932,7.414544,...,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565
SJBALL013388,-1.440565,-1.440565,3.841757,2.745244,3.235896,2.133168,-1.021823,4.428643,4.836435,6.315107,...,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565
SJALL049970,-1.234093,-1.440565,4.239446,3.304800,3.978864,6.100735,0.673483,2.771707,4.053792,7.608850,...,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565
SJBALL020583,-0.676836,-1.440565,5.204849,2.044409,2.690508,3.750090,-1.325883,1.232367,3.967201,6.795964,...,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565
SJCOGALL010907,-1.440565,-1.440565,4.410238,3.583172,3.216805,3.766060,-1.197123,0.724345,3.153118,6.818268,...,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
AML044-D,-0.622221,-1.440565,4.120989,3.837666,4.139783,7.176423,3.806023,7.249054,5.212846,5.604079,...,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565
867_B,-1.025907,-1.440565,5.082728,5.219812,3.475098,6.906820,-1.371075,4.530678,4.696927,5.803775,...,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565
867_T,2.253271,-1.440565,5.089315,5.461956,3.176504,5.184371,2.376063,3.971144,4.895359,6.367729,...,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565,-1.440565


In [22]:
# Get least frequently ocurring class count
min_class_count = y_train.value_counts().min()
min_class_count

np.int64(5)

In [23]:
y_train = y_train.iloc[:, 0].values  # Convert to 1D numpy array
y_test = y_test.iloc[:, 0].values
y_test

array(['B-ALL, BCR::ABL1-like', 'B-ALL, MEF2D-r', 'B-ALL, DUX4-r',
       'B-ALL, ETV6::RUNX1', 'B-ALL, iAMP21', 'B-ALL, PAX5alt',
       'B-ALL, ETV6::RUNX1', 'B-ALL, hypodiploid', 'B-ALL, PAX5alt',
       'B-ALL, TCF3::PBX1', 'B-ALL, hypodiploid', 'B-ALL, BCR::ABL1-like',
       'B-ALL, MEF2D-r', 'B-ALL, TCF3::PBX1', 'B-ALL, iAMP21',
       'B-ALL, DUX4-r', 'B-ALL, BCR::ABL1', 'B-ALL, BCR::ABL1',
       'B-ALL, BCR::ABL1-like', 'B-ALL, KMT2A-r', 'B-ALL, ETV6::RUNX1',
       'B-ALL, ZNF384-r', 'B-ALL, ETV6::RUNX1', 'B-ALL, ETV6::RUNX1-like',
       'B-ALL, hypodiploid', 'B-ALL, BCR::ABL1-like',
       'B-ALL, BCR::ABL1-like', 'B-ALL, ETV6::RUNX1', 'B-ALL, BCR::ABL1',
       'B-ALL, PAX5 P80R', 'B-ALL, ETV6::RUNX1-like',
       'B-ALL, high hyperdiploid', 'B-ALL, ZNF384-r',
       'B-ALL, ETV6::RUNX1', 'B-ALL, BCR::ABL1-like', 'B-ALL, KMT2A-r',
       'B-ALL, ETV6::RUNX1', 'B-ALL, BCR::ABL1-like', 'B-ALL, PAX5alt',
       'B-ALL, KMT2A-r', 'B-ALL, BCR::ABL1-like', 'B-ALL, ETV6::RUNX1',

In [24]:
N_ITER=30
N_REPEATS=1 # WE WILL VALIDATE FINAL PARAMS USING MORE REPEATS

# ----------------------
# Pipeline: Variance filter -> RF feature selection -> RF classifier
# ----------------------
pipeline = Pipeline([
    ("variance_filter", VarianceThreshold(threshold=VARIANCE_THRESHOLD)), # remove near-constant genes
    ("feature_selection", SelectPercentile(percentile=FEATURE_SELECTION_PERCENTILE)),  # keep only top 10% of features
    ("rf", RandomForestClassifier(random_seed, n_jobs=-n_jobs))
])

# ----------------------
# Hyperparameter search space
# ----------------------
param_grid = {
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [None, 20, 50],
    'rf__max_features': ['sqrt', 'log2'],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 5],
    'rf__class_weight': [None, 'balanced']
}

# ----------------------
# Cross-validation + randomized search
# ----------------------
cv = RepeatedStratifiedKFold(n_splits=min_class_count,
                             n_repeats=N_REPEATS,
                             random_state=random_seed)

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=N_ITER,
    cv=cv,
    scoring="f1_weighted",   # good for multiclass imbalance
    n_jobs=-n_jobs,
    random_state=random_seed,
    verbose=1
)

search.fit(X_train, y_train)

print("Best hyperparameters:", search.best_params_)
print("Best CV (F1-weighted):", search.best_score_)
best_pipeline = search.best_estimator_

## SAVE EVERYTHING
# Save best hyperparameters
joblib.dump(search.best_params_, hyperparameters_path)

# Save features
# First get features that passed the variance threshold
variance_mask = best_pipeline.named_steps['variance_filter'].get_support()
remaining_features = X_train.columns[variance_mask]

# Then get features selected by SelectFromModel
selected_mask = best_pipeline.named_steps['feature_selection'].get_support()
selected_features = remaining_features[selected_mask]

joblib.dump(selected_features, selected_features_path)

# Save the whole pipeline
joblib.dump(best_pipeline, pipeline_path)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


Best hyperparameters: {'rf__n_estimators': 200, 'rf__min_samples_split': 10, 'rf__min_samples_leaf': 5, 'rf__max_features': 'sqrt', 'rf__max_depth': 50, 'rf__class_weight': 'balanced'}
Best CV (F1-weighted): 0.9069368812055462


['/home/mariya/Data/pallas/training/gex/gex.pipeline.pkl']

In [25]:
# Cross-validate again with more repeats and larger estimators
N_REPEATS=10

# Use just the selected features
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

# Read in "best_hyperparameters.pkl"
with open(hyperparameters_path, "rb") as f:
    best_params = joblib.load(f)

best_params = {k.replace('rf__', ''): v for k, v in best_params.items()}
best_params['n_estimators'] = FINAL_N_ESTIMATORS
print("Final model parameters:", best_params)
final_rf = RandomForestClassifier(**best_params,
                            n_jobs=n_jobs,
                            random_state=random_seed)

# Cross-validate again with more repeats
cv = RepeatedStratifiedKFold(n_splits=min_class_count,
                             n_repeats=N_REPEATS,
                             random_state=random_seed)

cv_scores = cross_val_score(final_rf,
                           X_train_selected,
                           y_train,
                           cv=cv,
                           scoring="f1_weighted",
                           n_jobs=-n_jobs,
                           verbose=1)

print(f"Cross-validation F1-weighted: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Train the final model on all training data
final_rf.fit(X_train_selected, y_train)

# Save the final model
joblib.dump(final_rf, model_path)

Final model parameters: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'max_depth': 50, 'class_weight': 'balanced'}


[Parallel(n_jobs=-10)]: Using backend LokyBackend with 5 concurrent workers.
[Parallel(n_jobs=-10)]: Done  40 tasks      | elapsed:  8.5min
[Parallel(n_jobs=-10)]: Done  50 out of  50 | elapsed: 10.2min finished


Cross-validation F1-weighted: 0.914 ± 0.011


['/home/mariya/Data/pallas/training/gex/gex.model.pkl']

In [26]:
# ----------------------
# Evaluate final model on test set
# ----------------------

# Use selected features for test set
y_pred_final = final_rf.predict(X_test_selected)

# Print size of selected features
print(f"Number of selected features: {len(selected_features)}")

# Print shape of X_test_selected
print(f"Shape of X_test_selected: {X_test_selected.shape}")

print("\nTest set classification report (final optimized model):")
print(classification_report(y_test, y_pred_final, digits=3))


Number of selected features: 3921
Shape of X_test_selected: (767, 3921)

Test set classification report (final optimized model):
                          precision    recall  f1-score   support

      AML, APL-PML::RARA      1.000     1.000     1.000         4
     AML, CBFA2T3::GLIS2      0.857     0.857     0.857         7
        AML, CBFB::MYH11      1.000     1.000     1.000        34
           AML, CEBPAalt      1.000     0.857     0.923        14
        AML, DEK::NUP214      1.000     1.000     1.000         7
             AML, ETV6-r      1.000     0.500     0.667         2
           AML, FUS::ERG      1.000     1.000     1.000         1
            AML, KAT6A-r      1.000     0.500     0.667         2
            AML, KMT2A-r      0.875     0.969     0.920        65
            AML, MECOM-r      1.000     1.000     1.000         1
            AML, NPM1alt      0.897     0.897     0.897        29
            AML, NUP98-r      0.938     0.789     0.857        19
        AML,

In [27]:

# ----------------------
# Extract top genes from final model
# ----------------------
# Get feature importances from final RF
importances = final_rf.feature_importances_

# Rank genes by importance
gene_importances = pd.DataFrame({
    "Gene": selected_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

gt = GeneThesaurus()
translated = gt.translate_genes(gene_importances["Gene"].tolist(),
                               source="ensembl_id", target="symbol")
gene_importances["Gene"] = gene_importances["Gene"].map(translated)

print("\nTop 20 genes driving classification (final model):")
print(gene_importances.head(20))


Top 20 genes driving classification (final model):
         Gene  Importance
1773   MEGF10    0.003717
3080    NUTM1    0.003651
1715    MEIS1    0.002910
3262    GREB1    0.002632
3199   SOWAHD    0.002453
336   RUNX1T1    0.002319
131     PREX2    0.002290
2478     EVPL    0.002258
3721    ANXA8    0.002251
2038   TSPAN7    0.002149
2330     UNCX    0.002147
1195    KRT17    0.002132
3097   ANKS1B    0.002100
2068     DGKI    0.002055
332     HOXA9    0.002054
507       EFS    0.002042
3891  EXOC3L2    0.002010
2118  IGF2BP1    0.001999
3125     PBX1    0.001916
2920  LDLRAD3    0.001882


In [28]:
# How many of the feature importances are zero?
num_zero_importances = np.sum(importances == 0)
print(f"\nNumber of features with zero importance: {num_zero_importances} out of {len(importances)}")

# Print lowest importance genes as well
print("\nBottom 20 genes by importance (final model):")
print(gene_importances.tail(20))


Number of features with zero importance: 1 out of 3921

Bottom 20 genes by importance (final model):
              Gene    Importance
3885      C13orf46  6.813729e-06
3736           NaN  6.432677e-06
3591           NaN  6.160763e-06
3768      C17orf50  6.004831e-06
3781           NaN  5.644085e-06
3642          HTD2  5.063249e-06
3756           NaN  5.036921e-06
3782           NaN  4.919007e-06
3606      KRTAP5-7  4.906541e-06
935            SPR  4.040029e-06
3442        XAGE1A  3.967596e-06
3674    BUB1B-PAK6  3.421750e-06
984           GHRH  3.218229e-06
3882        ADORA3  2.901495e-06
1847       SCGB1A1  2.516179e-06
3779           NaN  2.324245e-06
3676           NaN  1.944050e-06
3886         TEX54  1.670785e-06
3671           NaN  1.051970e-07
3658  HOXA10-HOXA9  0.000000e+00
